# SPL Featurize Output Validation
Validates the featurized training data from `spl_pipeline_featurize.py` on branch `feature/B-2760706`.

## 1. Environment Setup
**Important:** Set env vars before any imports. Do NOT use `%env` — set them in terminal before starting the kernel, or use `os.environ` before imports.

In [ ]:
import os
os.environ['ENV_FOR_DYNACONF'] = 'prod'
os.environ['DYNACONF_GIT_BRANCH'] = 'feature/B-2760706'

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import DataFrame
from ltv_helpers.spark import create_spark
from classic_spl_ltv.config.paths import paths as p

## 2. Create Spark Session

In [ ]:
spark = create_spark(
    buckets=p.buckets,
    app_name='spl_featurize_validation',
    app_config_prod={
        'spark.driver.memory': '8g',
        'spark.sql.shuffle.partitions': '200',
        'spark.driver.maxResultSize': '4g',
    },
)

## 3. Load Featurized Training Data
Use Spark — do NOT load full parquet into pandas (will crash the kernel).

In [ ]:
# Training data path from paths.toml
train_path = 's3a://' + p.ret_ltv_training
print(f'Loading from: {train_path}')

df_train = spark.read.parquet(train_path)
print(f'Row count: {df_train.count()}')

## 4. Validate Schema & Columns

In [ ]:
# Print full schema
df_train.printSchema()

In [ ]:
columns = df_train.columns
print(f'Total columns: {len(columns)}')
print()
print('All columns:')
for col in sorted(columns):
    print(f'  {col}')

## 5. Check: No Autoregressive Columns
Yinan confirmed autoregressive columns should be removed.

In [ ]:
# Autoregressive columns typically contain 'autoreg' or 'ar_' in their name
# Adjust the pattern if your repo uses a different naming convention
auto_cols = [c for c in columns if 'autoreg' in c.lower() or 'ar_' in c.lower()]

if auto_cols:
    print('WARNING: Autoregressive columns still present!')
    for c in auto_cols:
        print(f'  {c}')
else:
    print('PASS: No autoregressive columns found.')

## 6. Check: Only SPL Lines Present
Expected lines: E16, E12, E72, Q98

In [ ]:
expected_lines = {'E16', 'E12', 'E72', 'Q98'}

# Adjust column name if needed — check schema output above
actual_lines = set(
    row['drv_line'] for row in df_train.select('drv_line').distinct().collect()
)

print(f'Expected lines: {sorted(expected_lines)}')
print(f'Actual lines:   {sorted(actual_lines)}')
print()

if actual_lines == expected_lines:
    print('PASS: Exactly the expected SPL lines are present.')
elif actual_lines.issubset(expected_lines):
    print(f'WARNING: Missing lines: {expected_lines - actual_lines}')
else:
    print(f'WARNING: Unexpected lines: {actual_lines - expected_lines}')

## 7. Check: Key Derived Columns Exist
`drv_ifs` and `drv_chnl_bnd` should be present.

In [ ]:
expected_cols = ['drv_ifs', 'drv_chnl_bnd']

for col in expected_cols:
    if col in columns:
        null_count = df_train.filter(df_train[col].isNull()).count()
        total = df_train.count()
        print(f'PASS: {col} exists | nulls: {null_count}/{total} ({null_count/total*100:.1f}%)')
    else:
        print(f'MISSING: {col} not found in output')

## 8. Row Counts by Line

In [ ]:
df_train.groupBy('drv_line').count().orderBy('drv_line').show()

## 9. Sample Data Inspection
Load a small sample into pandas for visual inspection.

In [ ]:
df_sample = df_train.limit(500).toPandas()
df_sample.head(10)

In [ ]:
# Basic stats on numeric columns
df_sample.describe()

## 10. Value Range Checks

In [ ]:
# drv_ifs: should be actual IFS values or -10 (sentinel for nulls)
df_train.select('drv_ifs').describe().show()
print('drv_ifs value distribution (top 10):')
df_train.groupBy('drv_ifs').count().orderBy('count', ascending=False).show(10)

In [ ]:
# drv_chnl_bnd: should be 'Agent', 'CCC', or 'Web'
print('drv_chnl_bnd value distribution:')
df_train.groupBy('drv_chnl_bnd').count().orderBy('count', ascending=False).show()

## 11. Cleanup

In [ ]:
spark.stop()